# MET-type taxonomy

Register the shared VISp MET-type hierarchy with plotting colors. Dataset assignments belong in Patch-seq and WNM ETLs.

In [1]:
import json
from pathlib import Path

import pandas as pd
import polars as pl

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models
from connects_common_connectivity.models import (
    AlgorithmRun,
    Cluster,
    ClusterHierarchy,
    HierarchyCategory,
)

In [2]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
COLOR_PATH = REPO_ROOT / "data/MET-type colors/met_type_colors.json"
TASIC_PATH = REPO_ROOT / "data/T-type colors Tasic/anno.feather"
OUTPUT_ROOT = Path(get_settings().output_root)

HIERARCHY_ID = "visp_met_types_taxonomy"
RUN_ID = "visp_met_types_clustering"
ROOT_ID = "cell"
for path in (COLOR_PATH, TASIC_PATH):
    assert path.is_file(), f"Missing local input: {path}"

## Shape hierarchy and colors

In [3]:
with COLOR_PATH.open() as stream:
    met_colors = json.load(stream)
assert all(isinstance(label, str) and isinstance(color, str) for label, color in met_colors.items())

# Shared class colors keep MET plots visually aligned with the T-type taxonomy.
tasic = pd.read_feather(TASIC_PATH)
class_colors = dict(zip(tasic["class_label"], tasic["class_color"]))
gaba_types = sorted(label for label in met_colors if "MET" in label)
glut_types = sorted(label for label in met_colors if "MET" not in label)
assert set(gaba_types).isdisjoint(glut_types)
assert (len(gaba_types), len(glut_types), len(met_colors)) == (28, 17, 45)

In [4]:
categories = [
    HierarchyCategory(id="cluster", hierarchy_id=HIERARCHY_ID, level=0, description="VISp MET-type leaf."),
    HierarchyCategory(id="class", hierarchy_id=HIERARCHY_ID, level=1, description="Excitatory or inhibitory class."),
    HierarchyCategory(id="major_class", hierarchy_id=HIERARCHY_ID, level=2, description="Synthetic taxonomy root."),
]
run = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="VISp morphology-electrophysiology-transcriptomics taxonomy",
    algorithm_version="2021",
)
classes = ["GABAergic", "Glutamatergic"]
leaves_by_class = {"GABAergic": gaba_types, "Glutamatergic": glut_types}
clusters = [
    Cluster(
        id=ROOT_ID, hierarchy_id=HIERARCHY_ID, children=classes,
        level=0, hex_color="#000000", hierarchy_category="major_class",
    )
]
clusters.extend(
    Cluster(
        id=class_id, hierarchy_id=HIERARCHY_ID, parent=ROOT_ID,
        children=leaves_by_class[class_id], level=1,
        hex_color=class_colors[class_id], hierarchy_category="class",
    )
    for class_id in classes
)
clusters.extend(
    Cluster(
        id=label, hierarchy_id=HIERARCHY_ID, parent=class_id,
        children=[], level=2, hex_color=met_colors[label],
        hierarchy_category="cluster",
    )
    for class_id in classes
    for label in leaves_by_class[class_id]
)
hierarchy = ClusterHierarchy(
    id=HIERARCHY_ID, run=RUN_ID, root=ROOT_ID,
    clusters=[cluster.id for cluster in clusters],
)
assert all(cluster.hex_color is not None for cluster in clusters)

## Write shared taxonomy

In [5]:
print(write_models([run], output_root=OUTPUT_ROOT))
print(write_models(categories, output_root=OUTPUT_ROOT))
print(write_models(clusters, output_root=OUTPUT_ROOT))
print(write_models([hierarchy], output_root=OUTPUT_ROOT))

WrittenResult(class_name='AlgorithmRun', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/algorithmrun'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=1)
WrittenResult(class_name='HierarchyCategory', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/hierarchycategory'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'visp_met_types_taxonomy\')',), rows_written=3)
WrittenResult(class_name='Cluster', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/cluster'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'visp_met_types_taxonomy\')',), rows_written=48)
WrittenResult(class_name='ClusterHierarchy', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_rel

In [6]:
# Verify complete colors and references after all taxonomy rows are persisted.
stored_clusters = pl.read_delta(OUTPUT_ROOT / "cluster").filter(
    pl.col("hierarchy_id") == HIERARCHY_ID
)
stored_hierarchy = pl.read_delta(OUTPUT_ROOT / "clusterhierarchy").filter(
    pl.col("id") == HIERARCHY_ID
)
stored_categories = pl.read_delta(OUTPUT_ROOT / "hierarchycategory").filter(
    pl.col("hierarchy_id") == HIERARCHY_ID
)
assert stored_clusters["hex_color"].null_count() == 0
assert stored_categories.height == 3
assert stored_hierarchy.height == 1
assert stored_hierarchy.item(0, "root") == ROOT_ID
assert set(stored_hierarchy.item(0, "clusters")) == set(stored_clusters["id"])
assert set(stored_clusters["parent"].drop_nulls()) <= set(stored_clusters["id"])
print("Colored MET-type taxonomy verified.")

Colored MET-type taxonomy verified.
